In [1]:
import pandas as pd
import numpy as np
import joblib

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
df = pd.read_csv("../data/raw/demand_raw.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (47, 9)

Columns:
['date', 'crop', 'location', 'demand_kg', 'modal_price', 'day', 'month', 'day_of_week', 'previous_demand']


,date,crop,location,demand_kg,modal_price,day,month,day_of_week,previous_demand
0,2026-04-05,Apple,Guntur,5.0,120.0,5,4,6,6.0
1,2026-06-18,Apple,Guntur,4.0,120.0,18,6,3,5.0
2,2026-08-05,Apple,Guntur,8.0,120.0,5,8,2,4.0
3,2026-08-18,Apple,Guntur,4.0,120.0,18,8,1,8.0
4,2026-09-05,Apple,Guntur,5.0,120.0,5,9,5,4.0


In [3]:
np.random.seed(42)

records = []

for _, row in df.iterrows():

    for _ in range(25):

        quantity = np.random.randint(100, 2001)

        production_cost_per_kg = np.random.uniform(
            row["modal_price"] * 0.45,
            row["modal_price"] * 0.75
        )

        selling_price_per_kg = row["modal_price"] * np.random.uniform(
            0.85, 1.05
        )

        transport_cost = np.random.uniform(300, 2500)

        other_cost = np.random.uniform(100, 800)

        revenue = quantity * selling_price_per_kg

        total_cost = (
            quantity * production_cost_per_kg
            + transport_cost
            + other_cost
        )

        profit = revenue - total_cost

        records.append({
            "crop": row["crop"],
            "location": row["location"],
            "demand_kg": row["demand_kg"],
            "modal_price": row["modal_price"],
            "previous_demand": row["previous_demand"],
            "month": row["month"],
            "quantity_kg": quantity,
            "production_cost_per_kg": production_cost_per_kg,
            "selling_price_per_kg": selling_price_per_kg,
            "transport_cost": transport_cost,
            "other_cost": other_cost,
            "profit": profit
        })

profit_df = pd.DataFrame(records)

print("Profit dataset shape:", profit_df.shape)
profit_df.head()

Profit dataset shape: (1175, 12)


,crop,location,demand_kg,modal_price,previous_demand,month,quantity_kg,production_cost_per_kg,selling_price_per_kg,transport_cost,other_cost,profit
0,Apple,Guntur,5.0,120.0,6.0,4,1226,82.675548,106.402435,2015.320201,517.795111,26556.048678
1,Apple,Guntur,5.0,120.0,6.0,4,221,59.615803,103.394007,2205.587521,520.780508,6948.615046
2,Apple,Guntur,5.0,120.0,6.0,4,1787,77.431985,103.353878,1888.397299,756.986896,43677.038364
3,Apple,Guntur,5.0,120.0,6.0,4,485,60.545699,106.401708,969.332935,467.329502,20803.502131
4,Apple,Guntur,5.0,120.0,6.0,4,847,54.830247,114.594592,1179.694138,132.665964,49308.039727


In [4]:
profit_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1175 entries, 0 to 1174
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   crop                    1175 non-null   object 
 1   location                1175 non-null   object 
 2   demand_kg               1175 non-null   float64
 3   modal_price             1175 non-null   float64
 4   previous_demand         1175 non-null   float64
 5   month                   1175 non-null   int64  
 6   quantity_kg             1175 non-null   int64  
 7   production_cost_per_kg  1175 non-null   float64
 8   selling_price_per_kg    1175 non-null   float64
 9   transport_cost          1175 non-null   float64
 10  other_cost              1175 non-null   float64
 11  profit                  1175 non-null   float64
dtypes: float64(8), int64(2), object(2)
memory usage: 110.3+ KB


In [5]:
profit_df[[
    "quantity_kg",
    "production_cost_per_kg",
    "selling_price_per_kg",
    "transport_cost",
    "other_cost",
    "profit"
]].describe()

,quantity_kg,production_cost_per_kg,selling_price_per_kg,transport_cost,other_cost,profit
count,1175.000000,1175.000000,1175.000000,1175.000000,1175.000000,1175.000000
mean,1048.129362,34.176483,54.194128,1405.245586,445.543782,19201.906878
std,553.958938,19.268455,29.370327,640.708336,201.994005,18611.320196
min,100.000000,12.688311,23.833863,300.499484,102.342453,-2107.439985
25%,578.000000,19.041957,30.491382,849.354445,279.450435,6202.842453
50%,1037.000000,25.238147,37.567436,1437.860741,442.616932,13823.023285
75%,1532.000000,46.974475,77.003701,1964.564384,620.695998,25624.048126
max,2000.000000,89.989836,125.919284,2498.996621,797.766426,104641.738133


In [6]:
print("Average profit:", profit_df["profit"].mean())
print("Minimum profit:", profit_df["profit"].min())
print("Maximum profit:", profit_df["profit"].max())

Average profit: 19201.906877727986
Minimum profit: -2107.439985423305
Maximum profit: 104641.73813312933


In [7]:
profit_df.to_csv(
    "../data/raw/farmer_profit_raw.csv",
    index=False
)

print("farmer_profit_raw.csv created successfully!")

farmer_profit_raw.csv created successfully!


In [8]:
X = profit_df.drop(columns=["profit"])
y = profit_df["profit"]

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print("profit")

Features:
['crop', 'location', 'demand_kg', 'modal_price', 'previous_demand', 'month', 'quantity_kg', 'production_cost_per_kg', 'selling_price_per_kg', 'transport_cost', 'other_cost']

Target:
profit


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = ["crop", "location"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

X_processed = preprocessor.fit_transform(X)

print("Preprocessing completed!")
print("Processed shape:", X_processed.shape)

Preprocessing completed!
Processed shape: (1175, 26)


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 940
Testing samples: 235


In [14]:
!{sys.executable} -m pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.8/101.7 MB 2.8 MB/s eta 0:00:37
    --------------------------------------- 1.6/101.7 MB 3.1 MB/s eta 0:00:33
    --------------------------------------- 2.4/101.7 MB 3.4 MB/s eta 0:00:30
   - -------------------------------------- 3.4/101.7 MB 3.7 MB/s eta 0:00:27
   - -------------------------------------- 4.5/101.7 MB 3.9 MB/s eta 0:00:25
   -- ------------------------------------- 5.5/101.7 MB 4.1 MB/s eta 0:00:24
   -- ------------------------------------- 6.3/101.7 MB 4.1 MB/s eta 0:00:24
   -- ------------------------------------- 7.6/101.7 MB 4.2 MB/s eta 0:00:23
   --- ------------------------------------ 9.2/101.7 MB 4.6 MB/s eta 0:00:21
   ---- ----------------------------------- 10.2/101.7 MB 4.7 MB/s eta 0:00:20
   ---- ----------------------------------- 11.5/101.7 MB 4.8 MB/s eta 0:00:1


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

model.fit(X_train, y_train)

print("XGBoost training completed!")

XGBoost training completed!


In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("Model Performance")
print("------------------")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

Model Performance
------------------
MAE : 1865.6111090300476
RMSE: 3172.289671032738
R²  : 0.9732067342436328


In [17]:
processed_df = profit_df.copy()

processed_df.to_csv(
    "../data/processed/farmer_profit_processed.csv",
    index=False
)

print("farmer_profit_processed.csv created!")

farmer_profit_processed.csv created!


In [18]:
joblib.dump(
    model,
    "../models/farmer_profit_model.pkl"
)

joblib.dump(
    preprocessor,
    "../models/profit_preprocessor.pkl"
)

print("Model saved successfully!")

Model saved successfully!


In [19]:
import numpy as np
import pandas as pd

np.random.seed(42)

periods = ["Week 1", "Week 2", "Week 3", "Week 4", "Week 5", "Week 6"]

traditional_profit = [
    3500, 3800, 3300, 4000, 3700, 4100
]

kisanmitra_profit = [
    4100, 4500, 4700, 5100, 5000, 5600
]

comparison_df = pd.DataFrame({
    "period": periods,
    "traditional_profit": traditional_profit,
    "kisanmitra_profit": kisanmitra_profit
})

comparison_df

,period,traditional_profit,kisanmitra_profit
0,Week 1,3500,4100
1,Week 2,3800,4500
2,Week 3,3300,4700
3,Week 4,4000,5100
4,Week 5,3700,5000
5,Week 6,4100,5600


In [20]:
comparison_df["improvement"] = (
    comparison_df["kisanmitra_profit"]
    - comparison_df["traditional_profit"]
)

comparison_df["improvement_percent"] = (
    comparison_df["improvement"]
    / comparison_df["traditional_profit"]
) * 100

comparison_df

,period,traditional_profit,kisanmitra_profit,improvement,improvement_percent
0,Week 1,3500,4100,600,17.142857
1,Week 2,3800,4500,700,18.421053
2,Week 3,3300,4700,1400,42.424242
3,Week 4,4000,5100,1100,27.500000
4,Week 5,3700,5000,1300,35.135135
5,Week 6,4100,5600,1500,36.585366


In [21]:
comparison_df.to_csv(
    "../data/processed/farmer_profit_comparison.csv",
    index=False
)

print("Comparison data saved!")

Comparison data saved!
